# Agente de Gestión de Inventario con MCP Remoto Personalizado

## Descripción

En este ejercicio, construirás un **sistema completo de agentes con servidor MCP remoto personalizado** para gestión de inventario aeronáutico. A diferencia del módulo 7 (MCP local con stdio), este módulo despliega el servidor MCP como un servicio HTTP/SSE que puede ser consultado remotamente.

### Arquitectura MCP Remoto Personalizado
```
Pregunta sobre inventario
    ↓
[InventoryAgent]
    ↓
[MCPStreamableHTTPTool]
    ↓
HTTP/SSE → http://localhost:8000/sse
    ↓
[Servidor MCP Remoto]
    ├→ get_inventory_levels()
    ├→ get_weekly_sales()
    └→ get_critical_items(threshold)
    ↓
Análisis y respuesta
```

### Ventajas de MCP Remoto
- **Escalabilidad**: El servidor puede estar en otra máquina/contenedor
- **Múltiples clientes**: Varios agentes pueden consultar el mismo servidor
- **Mantenimiento**: Actualizar el servidor sin modificar los clientes
- **GitHub Codespaces**: Ideal para desarrollo en la nube con puertos expuestos

### Diferencias con MCP Local (Módulo 7)
| Aspecto | MCP Local (Módulo 7) | MCP Remoto (Módulo 10) |
|---------|----------------------|------------------------|
| **Comunicación** | stdio (procesos) | HTTP/SSE (red) |
| **Tool** | `MCPStdioTool` | `MCPStreamableHTTPTool` |
| **Ejecución** | `python server.py` | Servidor HTTP en puerto 8000 |
| **Uso** | Desarrollo local simple | Producción/Cloud/Multi-cliente |
| **Latencia** | Muy baja (proceso local) | Ligeramente mayor (red) |

## Paso 1: Iniciar el Servidor MCP Remoto

**IMPORTANTE**: Antes de ejecutar este notebook, debes iniciar el servidor MCP remoto en una terminal separada:

```bash
cd Labfiles/10-agent-framework-mcp-remote-custom-es/Python
python remote_mcp_server.py
```

El servidor debe estar ejecutándose en `http://localhost:8000` antes de continuar.

### Verificar que el servidor está activo

Puedes verificar que el servidor está activo ejecutando:

In [ ]:
import requests

try:
    response = requests.get("http://localhost:8000/health", timeout=2)
    print("✅ Servidor MCP remoto está activo y disponible")
except requests.exceptions.RequestException:
    print("❌ Servidor MCP remoto NO está disponible")
    print("⚠️  Por favor, inicia el servidor con: python remote_mcp_server.py")

## Paso 2: Cargar librerías y variables de entorno

In [ ]:
import asyncio
import os
from dotenv import load_dotenv, find_dotenv
from agent_framework import ChatAgent, MCPStreamableHTTPTool
from agent_framework.openai import OpenAIChatClient

load_dotenv(find_dotenv(usecwd=True))

# GitHub Models client configuration
MODEL_NAME = os.getenv("GITHUB_MODEL", "openai/gpt-4o")

print(f"Modelo configurado: {MODEL_NAME}")
print(f"Servidor MCP remoto: http://localhost:8000/sse")

## Paso 3: Crear y ejecutar el agente cliente

El `ChatAgent` se conecta al servidor MCP remoto usando `MCPStreamableHTTPTool`, que establece una conexión HTTP/SSE al servidor en el puerto 8000.

In [ ]:
async def query_remote_inventory(prompt):
    print("Conectando al servidor MCP remoto...")
    
    # GitHub Models client configuration
    client = OpenAIChatClient(
        model_id=MODEL_NAME,
        api_key=os.environ["GITHUB_TOKEN"],
        base_url="https://models.github.ai/inference"
    )
    
    try:
        async with (
            MCPStreamableHTTPTool(
                name="aeroinventory_remote",
                url="http://localhost:8000/sse",
            ) as mcp_server,
            ChatAgent(
                chat_client=client,
                name="InventoryAgent",
                instructions=(
                    "Eres un asistente de gestión de inventario aeronáutico que puede consultar niveles de stock, "
                    "ventas semanales y componentes críticos mediante un servidor MCP remoto. "
                    "Analiza los datos y proporciona recomendaciones claras sobre gestión de inventario."
                ),
            ) as agent,
        ):
            print("✅ Conectado al servidor MCP remoto")
            print(f"🔍 Ejecutando consulta: {prompt}")
            result = await agent.run(prompt, tools=mcp_server)
            print("\n📊 Resultado de la consulta:")
            print(result)
            return result
    except Exception as e:
        print(f"❌ Error al conectar con el servidor MCP remoto: {e}")
        print("⚠️  Asegúrate de que el servidor esté ejecutándose: python remote_mcp_server.py")
        raise

## Ejemplo 1: Lista de componentes con inventario bajo

In [ ]:
# Consulta 1: Componentes con inventario < 15
print("=" * 80)
print("CONSULTA 1: Componentes con inventario bajo")
print("=" * 80)
user_prompt = "Lista los componentes con inventario < 15 y su consumo semanal."
await query_remote_inventory(user_prompt)
print("\n" + "=" * 80)

## Ejemplo 2: Análisis de riesgo de desabastecimiento

In [ ]:
# Consulta 2: Riesgo de desabastecimiento
print("=" * 80)
print("CONSULTA 2: Análisis de riesgo de desabastecimiento")
print("=" * 80)
user_prompt = "¿Qué piezas podrían agotarse esta semana considerando el consumo semanal y el inventario actual?"
await query_remote_inventory(user_prompt)
print("\n" + "=" * 80)

## Ejemplo 3: Uso de herramienta con parámetros

In [ ]:
# Consulta 3: Componentes críticos con umbral personalizado
print("=" * 80)
print("CONSULTA 3: Componentes críticos (umbral personalizado)")
print("=" * 80)
user_prompt = "Muéstrame los componentes con inventario menor a 20 unidades usando la herramienta get_critical_items."
await query_remote_inventory(user_prompt)
print("\n" + "=" * 80)

## Ejemplo 4: Análisis comparativo completo

In [ ]:
# Consulta 4: Análisis comparativo
print("=" * 80)
print("CONSULTA 4: Análisis comparativo completo")
print("=" * 80)
user_prompt = (
    "Dame un análisis completo del inventario: "
    "1) Componentes con mayor demanda vs inventario disponible, "
    "2) Prioridad de reorden basada en días de cobertura, "
    "3) Componentes con sobre-stock (inventario > 3x consumo semanal)."
)
await query_remote_inventory(user_prompt)
print("\n" + "=" * 80)

## Resumen y Comparación

### ¿Cuándo usar cada enfoque?

**MCP Local (Módulo 7)**:
- ✅ Desarrollo rápido y pruebas locales
- ✅ No requiere configuración de red
- ✅ Latencia mínima
- ❌ Un cliente por servidor
- ❌ Difícil de escalar

**MCP Remoto (Módulo 10)**:
- ✅ Múltiples clientes concurrentes
- ✅ Servidor en otra máquina/contenedor/cloud
- ✅ Fácil mantenimiento y actualización
- ✅ Ideal para GitHub Codespaces y producción
- ❌ Requiere gestión de puertos y red
- ❌ Ligeramente más complejo de configurar

### Próximos pasos

1. **Despliegue en Codespaces**: Exponer el puerto 8000 públicamente
2. **Seguridad**: Agregar autenticación con tokens o API keys
3. **Persistencia**: Conectar a base de datos real en lugar de datos estáticos
4. **Monitoring**: Agregar logs estructurados y métricas
5. **Dockerización**: Crear Dockerfile para despliegue en contenedores